# 🏥 Simulation-Based Synthetic Data — Emergency Room

In this lesson we will build a simulation model that generates realistic synthetic event data by recreating how a system behaves over time, instead of listing fixed rules.

Let's assume the following scenario, an urban hospital wants to stress-test a new patient management system, but sharing real patient records is off the table. We'll simulate one 8-hour ER shift and capture every event as a row of synthetic data: 

1. Model patient arrivals using a Poisson distribution
2. Define the patient journey through the ER as a sequence of stages
3. Build an `AllocationAgent` that decides which patient sees the next available doctor
4. Run the simulation and collect every state transition as an event log
5. Explore the resulting synthetic dataset

## 0. Setup

In [1]:
# Install required packages
# Run this cell once, then restart the kernel
!pip install -q mesa

## 1. The Scenario: One ER Shift

Every patient who walks into the ER follows the same journey:

```
ARRIVE ──► TRIAGE ──► WAITING ──► CONSULTATION ──► TREATMENT ──► DISCHARGED
            2–8 min    until a      severity-          severity-
                       doctor       dependent          dependent
                       is free
```

### Business Assumptions
Before running a simulation, you have to make explicit choices about how the system works. These are the assumptions baked into this model:

| Assumption | Value | Rationale |
|---|---|---|
| Shift duration | 8 hours | Standard ER day shift |
| Doctors on duty | 3 | Small urban ER, daytime staffing |
| Arrival rate | ~7 patients/hour | Moderate load; busy but not overwhelmed |
| Severity distribution | 40% minor → 5% critical | Reflects typical ER case mix |
| Triage duration | 2–8 minutes | Uniform — quick and consistent |
| Consultation duration | Scales with severity | Severity 1: ~12 min · Severity 5: ~28 min |
| Treatment duration | Scales with severity | Severity 1: ~20 min · Severity 5: ~60 min |

Changing any of these changes the synthetic data. A simulation with 6 doctors and low arrival rates produces near-zero wait times; the same arrival rate with 2 doctors generates chronic queuing. **The assumptions are the model**.

### Two sources of variability

- **Arrival timing** follows a **Poisson process** — patients don't arrive at fixed intervals; they cluster unpredictably, just like in real life.
- **Stage durations** are drawn from probability distributions, not fixed values — so no two patients take exactly the same path even at the same severity level.

The key resource constraint in this scenario is the **doctor pool**.

In [3]:
import numpy as np
import pandas as pd
from mesa import Agent, Model

# ── Simulation parameters ───────────────────────────────────────────────
RANDOM_SEED    = 42
SIM_HOURS      = 8      # number of hours of an ER shift
STEPS_PER_HOUR = 60     # 1 step = 1 minute
TOTAL_STEPS    = SIM_HOURS * STEPS_PER_HOUR   # 480 minutes
NUM_DOCTORS    = 3      # Number of doctors per shift in an ER
ARRIVAL_RATE   = 0.12   # expected patients per minute  (~7 per hour)

# Severity: 1 = minor, 5 = critical  (skewed — most cases are non-critical)
SEVERITY_LEVELS  = [1, 2, 3, 4, 5]
SEVERITY_WEIGHTS = [0.40, 0.30, 0.15, 0.10, 0.05]

print("Configuration ready ✓")
print(f"  Shift duration : {SIM_HOURS} hours ({TOTAL_STEPS} steps)")
print(f"  Doctors on duty: {NUM_DOCTORS}")
print(f"  Expected arrivals: ~{int(ARRIVAL_RATE * STEPS_PER_HOUR * SIM_HOURS)} patients over the shift")

Configuration ready ✓
  Shift duration : 8 hours (480 steps)
  Doctors on duty: 3
  Expected arrivals: ~57 patients over the shift


## 2. The Patient

Each patient is a plain Python object carrying their attributes and current stage. When a patient's stage changes, the model records the event, that record becomes a row in our synthetic dataset.

In [4]:
class Patient:
    """A patient moving through the ER stages."""

    def __init__(self, patient_id, arrival_time, severity, rng):
        self.patient_id   = f"PT-{patient_id:04d}"
        self.arrival_time = arrival_time
        self.severity     = severity
        self.stage        = "triage"
        self.stage_start  = arrival_time
        self.assigned_doc = None
        self.wait_minutes = None

        # Stage durations drawn from distributions, scaled by severity
        self.triage_duration       = int(rng.uniform(2, 8))
        self.consultation_duration = max(5, int(rng.normal(8 + severity * 4, 3)))
        self.treatment_duration    = max(5, int(rng.exponential(10 + severity * 10)))

    def __repr__(self):
        return f"Patient({self.patient_id}, severity={self.severity}, stage={self.stage})"


# Patient quick demo
rng_demo = np.random.default_rng(0)
print("Example patients:")
for sev in [1, 3, 5]:
    p = Patient(0, arrival_time=0, severity=sev, rng=rng_demo)
    print(f"  Severity {sev} — triage {p.triage_duration} min, "
          f"consultation {p.consultation_duration} min, "
          f"treatment {p.treatment_duration} min")

Example patients:
  Severity 1 — triage 5 min, consultation 11 min, treatment 5 min
  Severity 3 — triage 2 min, consultation 18 min, treatment 65 min
  Severity 5 — triage 5 min, consultation 30 min, treatment 169 min


## 3. The Allocation Agent

Whenever a doctor becomes available, someone has to decide which waiting patient goes next. That decision belongs to the **`AllocationAgent`**.

In this notebook, the agent uses a simple rule: prioritize the most severe patient; break ties by longest wait. This is already more realistic than a pure FIFO queue as it mirrors how a real charge nurse would triage the waiting room.

> **Active Learning opportunity:** The `AllocationAgent` currently follows a fixed rule. Could an LLM make better allocation decisions — weighing queue length, patient severity, and doctor workload all at once? Try replacing `decide()` with an LLM call and see how the wait times change.

In [8]:
class AllocationAgent(Agent):
    """
    Decides which patient from the waiting queue should be seen next.
    """

    def __init__(self, model):
        super().__init__(model)

    def decide(self, waiting_queue, current_time):
        """Return the highest-priority patient, or None if the queue is empty."""
        if not waiting_queue:
            return None
        return max(
            waiting_queue,
            key=lambda p: (p.severity, current_time - p.stage_start)
        )

print("AllocationAgent defined ✓")
print("  Strategy: highest severity → longest wait")

AllocationAgent defined ✓
  Strategy: highest severity → longest wait


## 4. The ER Model

The `ERModel` is the simulation environment. It owns time, the doctor pool, the patient queues, and the event log. Every step (one minute of simulated time) it:

1. Draws new arrivals from a Poisson distribution
2. Advances every active patient through their current stage
3. Asks the `AllocationAgent` to assign waiting patients to free doctors
   
Note: every time a patient's stage changes, we write a row to the event log. That log is our synthetic dataset!

In [9]:
class ERModel(Model):
    """The ER environment. Time advances in 1-minute steps."""

    def __init__(self, num_doctors=NUM_DOCTORS, arrival_rate=ARRIVAL_RATE, seed=RANDOM_SEED):
        super().__init__()
        self.rng          = np.random.default_rng(seed)
        self.arrival_rate = arrival_rate
        self.current_step = 0
        self._next_id     = 0

        # Doctor pool — True = available
        self.doctors = {f"DR-{i+1:02d}": True for i in range(num_doctors)}

        # Patient queues
        self.in_triage       = []
        self.waiting         = []
        self.in_consultation = []
        self.in_treatment    = []

        # The decision-maker
        self.agent = AllocationAgent(model=self)

        # Every stage transition becomes one row here
        self.event_log = []

    # ── Doctor management ──────────────────────────────────────────────────
    def _free_doctor(self):
        for doc_id, available in self.doctors.items():
            if available:
                self.doctors[doc_id] = False
                return doc_id
        return None

    def _release_doctor(self, doc_id):
        self.doctors[doc_id] = True

    # ── Event logging ──────────────────────────────────────────────────────
    def _record(self, patient):
        self.event_log.append({
            "patient_id"  : patient.patient_id,
            "stage"       : patient.stage,
            "time_min"    : self.current_step,
            "hour"        : round(self.current_step / 60, 2),
            "severity"    : patient.severity,
            "doctor_id"   : patient.assigned_doc,
            "wait_min"    : patient.wait_minutes,
        })

    # ── One simulation step ────────────────────────────────────────────────
    def step(self):
        t = self.current_step

        # 1. New arrivals — Poisson(λ) per step
        n_arrivals = self.rng.poisson(self.arrival_rate)
        for _ in range(n_arrivals):
            severity = self.rng.choice(SEVERITY_LEVELS, p=SEVERITY_WEIGHTS)
            p = Patient(self._next_id, arrival_time=t, severity=severity, rng=self.rng)
            self._next_id += 1
            self.in_triage.append(p)
            self._record(p)   # log stage = "triage"

        # 2. Advance triage → waiting
        still_triaging = []
        for p in self.in_triage:
            if (t - p.stage_start) >= p.triage_duration:
                p.stage = "waiting"
                p.stage_start = t
                self.waiting.append(p)
                self._record(p)
            else:
                still_triaging.append(p)
        self.in_triage = still_triaging

        # 3. Allocation agent assigns waiting patients to free doctors
        while self.waiting:
            next_patient = self.agent.decide(self.waiting, t)
            if next_patient is None:
                break
            doc = self._free_doctor()
            if doc is None:
                break   # all doctors busy — patients keep waiting
            self.waiting.remove(next_patient)
            next_patient.assigned_doc = doc
            next_patient.wait_minutes = t - next_patient.stage_start
            next_patient.stage        = "consultation"
            next_patient.stage_start  = t
            self.in_consultation.append(next_patient)
            self._record(next_patient)

        # 4. Advance consultation → treatment
        still_consulting = []
        for p in self.in_consultation:
            if (t - p.stage_start) >= p.consultation_duration:
                p.stage = "treatment"
                p.stage_start = t
                self.in_treatment.append(p)
                self._record(p)
            else:
                still_consulting.append(p)
        self.in_consultation = still_consulting

        # 5. Advance treatment → discharged
        still_treating = []
        for p in self.in_treatment:
            if (t - p.stage_start) >= p.treatment_duration:
                self._release_doctor(p.assigned_doc)
                p.stage = "discharged"
                p.stage_start = t
                self._record(p)
            else:
                still_treating.append(p)
        self.in_treatment = still_treating

        self.current_step += 1

print("ERModel defined ✓")

ERModel defined ✓


## 6. Run the Simulation & create our synthetic dataset

In [10]:
er = ERModel()

print(f"Simulating {SIM_HOURS}-hour ER shift ({TOTAL_STEPS} steps)...\n")
print(f"{'Hour':>5}  {'Active':>8}  {'Waiting':>8}  {'Discharged':>11}  {'Events':>7}")
print("-" * 46)

for step in range(TOTAL_STEPS):
    er.step()

    if (step + 1) % STEPS_PER_HOUR == 0:
        hour       = (step + 1) // STEPS_PER_HOUR
        active     = len(er.in_triage) + len(er.waiting) + len(er.in_consultation) + len(er.in_treatment)
        waiting    = len(er.waiting)
        discharged = sum(1 for e in er.event_log if e["stage"] == "discharged")
        print(f"{hour:>4}h  {active:>8}  {waiting:>8}  {discharged:>11}  {len(er.event_log):>7}")

print("-" * 46)
print(f"\n✓ Simulation complete")
print(f"  {er._next_id} patients arrived over {SIM_HOURS} hours")
print(f"  {len(er.event_log):,} events recorded → your synthetic dataset")

Simulating 8-hour ER shift (480 steps)...

 Hour    Active   Waiting   Discharged   Events
----------------------------------------------
   1h         0         0            4       20
   2h         6         1            5       40
   3h        11         7            7       62
   4h        10         6           11       80
   5h        20        16           12      105
   6h        24        19           13      116
   7h        32        29           16      149
   8h        37        34           17      165
----------------------------------------------

✓ Simulation complete
  54 patients arrived over 8 hours
  165 events recorded → your synthetic dataset


In [12]:
df = pd.DataFrame(er.event_log)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Columns:", list(df.columns))
df.head(15)

Dataset shape: 165 rows × 7 columns

Columns: ['patient_id', 'stage', 'time_min', 'hour', 'severity', 'doctor_id', 'wait_min']


,patient_id,stage,time_min,hour,severity,doctor_id,wait_min
0,PT-0000,triage,5,0.08,3,None,NaN
1,PT-0001,triage,6,0.10,3,None,NaN
2,PT-0000,waiting,7,0.12,3,None,NaN
3,PT-0000,consultation,7,0.12,3,DR-01,0.0
4,PT-0001,waiting,10,0.17,3,None,NaN
5,PT-0001,consultation,10,0.17,3,DR-02,0.0
6,PT-0002,triage,12,0.20,3,None,NaN
7,PT-0002,waiting,15,0.25,3,None,NaN
8,PT-0002,consultation,15,0.25,3,DR-03,0.0
9,PT-0003,triage,16,0.27,1,None,NaN


## ✅ Summary

In this notebook you:

1. **Modelled patient arrivals** using a Poisson distribution — capturing the unpredictable clustering of real ER traffic
2. **Defined the patient journey** as a sequence of stages with severity-scaled durations drawn from probability distributions
3. **Built an `AllocationAgent`** (a Mesa `Agent`) that decides which waiting patient sees the next available doctor — prioritising severity, then wait time
4. **Ran the simulation** step by step, watching patients flow through the system and the event log grow
5. **Collected the event log as a synthetic dataset** — realistic operational data that emerged from the system dynamics, not from explicit row-by-row rules